In [15]:
import os
import json
import requests
import pickle
import pandas as pd

In [2]:

# Define the GitHub API endpoint for the target folder
api_url = "https://api.github.com/repos/lascivaroma/latin-lemmatized-texts/contents/lemmatized/xml"

# Optional: add headers to increase rate limits if needed
headers = {"Accept": "application/vnd.github.v3+json"}

response = requests.get(api_url, headers=headers)

if response.status_code == 200:
    files = response.json()
    xml_files = [file["name"] for file in files if file["type"] == "file"]

In [3]:
xml_files

['urn:cts:greekLit:tlg0031.tlg001.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg002.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg003.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg004.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg005.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg006.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg007.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg008.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg009.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg010.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg011.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg012.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg013.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg014.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg015.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg016.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg017.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg018.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg019.obi-lat1.xml',
 'urn:cts:greekLit:tlg0031.tlg020.obi-lat1.xml',
 'urn:cts:greekLit:t

In [4]:
filenames_vulgate = [f for f in xml_files if (":tlg0031" in f) or (":tlg0527" in f)]
len(filenames_vulgate)

73

In [5]:
import requests
import re
from bs4 import BeautifulSoup

def map_tei_pos(pos_tag):
    POS_MAP_3LETTER = {
        "VER": "VERB",
        "NOM": "NOUN",
        "ADJ": "ADJ",
        "ADV": "ADV",
        "PRE": "ADP",
        "CON": "CCONJ",
        "PRO": "PRON",
    }

    if pos_tag == "NOMpro":
        return "PROPN"
    elif pos_tag == "ADJcar":
        return "NUM"
    elif pos_tag == "CONsub":
        return "SCONJ"
    elif pos_tag == "CONcoo":
        return "CCONJ"

    return POS_MAP_3LETTER.get((pos_tag or "")[:3], "X")

def parse_tei_verses(file_name):
    url = f"https://raw.githubusercontent.com/lascivaroma/latin-lemmatized-texts/main/lemmatized/xml/{file_name}"
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "xml")

    # Extract CTS URN and construct work_id
    urn = soup.find("text")["n"]
    work_id = urn.replace("urn:cts:greekLit:", "").replace(".xml", "").replace(".obi-lat1", ".obi-lat")

    # Extract book title from header
    title_tag = soup.find("title")
    title = title_tag.text.strip() if title_tag else "Unknown_Book"

    verses = []

    for n, ab in enumerate(soup.find_all("ab", {"type": "verse"})):
        verse_id = ab.get("n")  # e.g., urn:cts:greekLit:tlg0031.tlg003.obi-lat1:1.13
        chapter_verse = verse_id.split(":")[-1]
        chapter_verse_list = chapter_verse.split(".")  # e.g., 1.13
        ref = {"chapter": chapter_verse_list[0], "verse": chapter_verse_list[1]}

        token_data = []
        text_parts = []
        cur_pos = 0

        # Iterate over words and punctuation in document order
        for node in ab.find_all(["w", "pc"]):
            if node.name == "w":
                word = (node.text or "").strip()
                if not word:
                    continue
                lemma_raw = node.get("lemma") or ""
                lemma = re.sub(r"\d+$", "", lemma_raw)
                pos = map_tei_pos(node.get("pos"))

                # Add a space before words if there's already content
                if text_parts:
                    text_parts.append(" ")
                    cur_pos += 1

                start = cur_pos
                text_parts.append(word)
                cur_pos += len(word)
                end = cur_pos  # exclusive

                token_data.append(({"token_text": word, "lemma" : lemma, "pos" : pos, "ref" : ref, "char_start": start, "char_end" : end}))

            elif node.name == "pc":
                # Punctuation: attach without a leading space
                punct = (node.text or "").strip()
                if not punct:
                    continue
                # POS/lemma choice for punctuation
                word = punct
                lemma = punct
                pos = "PUNCT"

                start = cur_pos
                text_parts.append(punct)
                cur_pos += len(punct)
                end = cur_pos  # exclusive

                token_data.append(({"token_text": word, "lemma" : lemma, "pos" : pos, "ref" : ref, "char_start": start, "char_end" : end}))

        verse_text = "".join(text_parts)
        verses.append({"work_id" : work_id, "sent_id" : n, "sent_text" : verse_text, "token_data" : token_data})

    return work_id, title, verses

In [23]:
parse_tei_verses(filenames_vulgate[0])[2][:10]

[{'work_id': 'tlg0031.tlg001.obi-lat',
  'sent_id': 0,
  'sent_text': 'liber generationis Iesu Christi filii David filii Abraham',
  'token_data': [{'token_text': 'liber',
    'lemma': 'liber',
    'pos': 'ADJ',
    'ref': {'chapter': '1', 'verse': '1'},
    'char_start': 0,
    'char_end': 5},
   {'token_text': 'generationis',
    'lemma': 'generatio',
    'pos': 'NOUN',
    'ref': {'chapter': '1', 'verse': '1'},
    'char_start': 6,
    'char_end': 18},
   {'token_text': 'Iesu',
    'lemma': 'Iesus',
    'pos': 'PROPN',
    'ref': {'chapter': '1', 'verse': '1'},
    'char_start': 19,
    'char_end': 23},
   {'token_text': 'Christi',
    'lemma': 'Christus',
    'pos': 'PROPN',
    'ref': {'chapter': '1', 'verse': '1'},
    'char_start': 24,
    'char_end': 31},
   {'token_text': 'filii',
    'lemma': 'filius',
    'pos': 'NOUN',
    'ref': {'chapter': '1', 'verse': '1'},
    'char_start': 32,
    'char_end': 37},
   {'token_text': 'David',
    'lemma': 'Dauid',
    'pos': 'PROPN',
  

In [13]:
target_jsons_path = "../data/vulgate_sentences_dicts/"
os.makedirs(target_jsons_path, exist_ok=True)

In [24]:
vulgate_works = []
for filename in filenames_vulgate:
    work_id, title, verses = parse_tei_verses(filename)
    vulgate_works.append({"grela_id" : "vulgate" + "_" + work_id, "title" : "Vulgate - " + title})
    with open(target_jsons_path + str(work_id) + ".json", "w") as f:
        json.dump(verses, f)

In [25]:
# Save vulgate_works
with open("../data/vulgate_works.pkl", "wb") as f:
    pickle.dump(vulgate_works, f)

In [26]:
vulgate_works_df = pd.DataFrame(vulgate_works)
vulgate_works_df.head(5)

,grela_id,title
0,vulgate_tlg0031.tlg001.obi-lat,Vulgate - Matthew
1,vulgate_tlg0031.tlg002.obi-lat,Vulgate - Mark
2,vulgate_tlg0031.tlg003.obi-lat,Vulgate - Luke
3,vulgate_tlg0031.tlg004.obi-lat,Vulgate - John
4,vulgate_tlg0031.tlg005.obi-lat,Vulgate - Acts


In [27]:
vulgate_works_df.to_parquet("../data/vulgate_works_df.parquet")